# pbzarr cohort example: per-sample d4 → per-population mean depth

This notebook walks through a small end-to-end PBZ workflow:

1. Synthesize a few tiny per-sample `.d4` files with `d4tools`.
2. Create an empty `.pbz` store and import the d4s as columns of a cohort `depth` track.
3. Open the store with xarray, attach a `pop` coordinate to the `sample` dim.
4. Reduce with `groupby("pop").mean("sample")`.
5. Write the reduction back to the **same** store as a new track whose column dim is `pop`.

Requires the pixi `example` env so that `d4tools` + `ipykernel` are available: run `pixi run -e example jupyter lab` (or your editor's notebook runner) from the repo root.

In [1]:
import subprocess
import tempfile
from pathlib import Path

import numpy as np
import xarray as xr

import pbzarr

work = Path(tempfile.mkdtemp(prefix="pbz_example_"))
print(work)

/var/folders/jd/wspg2z057cn1ndxt_ctq8j6m0000gq/T/pbz_example_d17mgio5


## 1. Synthesize per-sample d4 files

Four samples split into two populations. One small contig (`chr1`, 1000 bp) is plenty for the demo.

In [2]:
samples = {
    "s1": "POP_A",
    "s2": "POP_A",
    "s3": "POP_B",
    "s4": "POP_B",
}
contig_name, contig_len = "chr1", 1000

chrom_sizes = work / "genome.sizes"
chrom_sizes.write_text(f"{contig_name}\t{contig_len}\n")

rng = np.random.default_rng(0)
d4_paths: dict[str, Path] = {}
for name in samples:
    bg = work / f"{name}.bedgraph"
    rows = [
        f"{contig_name}\t{start}\t{start + 100}\t{int(rng.integers(0, 50))}"
        for start in range(0, contig_len, 100)
    ]
    bg.write_text("\n".join(rows) + "\n")

    d4 = work / f"{name}.d4"
    subprocess.run(
        ["d4tools", "create", "-g", str(chrom_sizes), str(bg), str(d4)],
        check=True,
    )
    d4_paths[name] = d4

d4_paths

{'s1': PosixPath('/var/folders/jd/wspg2z057cn1ndxt_ctq8j6m0000gq/T/pbz_example_d17mgio5/s1.d4'),
 's2': PosixPath('/var/folders/jd/wspg2z057cn1ndxt_ctq8j6m0000gq/T/pbz_example_d17mgio5/s2.d4'),
 's3': PosixPath('/var/folders/jd/wspg2z057cn1ndxt_ctq8j6m0000gq/T/pbz_example_d17mgio5/s3.d4'),
 's4': PosixPath('/var/folders/jd/wspg2z057cn1ndxt_ctq8j6m0000gq/T/pbz_example_d17mgio5/s4.d4')}

## 2. Create the pbz store and import the d4s as a cohort track

`columns=` makes the track 2D (`position`, `sample`); `column_dim="sample"` is the convention for cohort framing.

In [3]:
store = str(work / "cohort.pbz")

pbzarr.create_store(
    store,
    contigs=[contig_name],
    contig_lengths=[contig_len],
)

pbzarr.create_track(
    store,
    track="depth",
    dtype="int32",
    columns=list(samples.keys()),
    column_dim="sample",
)

pbzarr.import_d4(
    store,
    "depth",
    [(str(path), name) for name, path in d4_paths.items()],
)

/Users/cade/dev/pbzarr-dev/pbzarr-rs/.pixi/envs/default/lib/python3.12/site-packages/zarr/api/asynchronous.py:231: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


## 3. Open with xarray and attach a `pop` coordinate

`pbzarr.open` returns an `xr.DataTree`; each contig is a child node. We pull `chr1` out as a `Dataset` and add `pop` as a non-dim coord on the `sample` axis.

In [4]:
dt = pbzarr.open(store)
ds = dt[contig_name].to_dataset()

ds = ds.assign_coords(
    pop=("sample", [samples[s] for s in ds["sample"].values])
)
ds

<xarray.Dataset> Size: 16kB
Dimensions:  (position: 1000, sample: 4, contigs: 1)
Coordinates:
  * sample   (sample) object 32B 's1' 's2' 's3' 's4'
  * contigs  (contigs) object 8B 'chr1'
    pop      (sample) <U5 80B 'POP_A' 'POP_A' 'POP_B' 'POP_B'
Dimensions without coordinates: position
Data variables:
    depth    (position, sample) int32 16kB ...

## 4. Reduce across samples by population

In [5]:
pop_mean = ds["depth"].groupby("pop").mean("sample")
pop_mean

<xarray.DataArray 'depth' (position: 1000, pop: 2)> Size: 16kB
array([[37. , 27.5],
       [37. , 27.5],
       [37. , 27.5],
       ...,
       [43. , 28.5],
       [43. , 28.5],
       [43. , 28.5]], shape=(1000, 2))
Coordinates:
  * pop      (pop) object 16B 'POP_A' 'POP_B'
Dimensions without coordinates: position

## 5. Write the reduction back as a new track

`pbzarr.write_track` infers the dtype, `column_dim`, and column labels from the DataArray, registers the new track, writes per-contig data, and refreshes consolidated metadata in one call.

In [6]:
pbzarr.write_track(
    store,
    "depth_pop_mean",
    {contig_name: pop_mean},
    description="Mean depth per population (reduction of `depth` over sample)",
)

/Users/cade/dev/pbzarr-dev/pbzarr-rs/.pixi/envs/default/lib/python3.12/site-packages/zarr/api/asynchronous.py:231: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


<Group file:///var/folders/jd/wspg2z057cn1ndxt_ctq8j6m0000gq/T/pbz_example_d17mgio5/cohort.pbz>

## 6. Reopen and verify both tracks coexist

In [7]:
dt2 = pbzarr.open(store)
print("tracks:", dt2.pbz.tracks)
dt2[contig_name].to_dataset()

tracks: ['depth', 'depth_pop_mean']


<xarray.Dataset> Size: 24kB
Dimensions:         (position: 1000, sample: 4, pop: 2, contigs: 1)
Coordinates:
  * sample          (sample) object 32B 's1' 's2' 's3' 's4'
  * pop             (pop) object 16B 'POP_A' 'POP_B'
  * contigs         (contigs) object 8B 'chr1'
Dimensions without coordinates: position
Data variables:
    depth           (position, sample) int32 16kB ...
    depth_pop_mean  (position, pop) float32 8kB ...

In [ ]:
expected = ds["depth"].groupby("pop").mean("sample").astype("float32")
got = dt2[contig_name]["depth_pop_mean"]
np.testing.assert_allclose(got.values, expected.transpose("position", "pop").values)
print("round-trip OK")